# TRPG 룰북 RAG

### 결합본 수정 (나중에 한 번에)
| 순서 | 위치 | 고칠 내용 | 이유 | 질문당 시간 |
|---|---|---|---|---|
| 1 | `NotebookRuleAdapter` | `results[0]["metadata"]["evidence_terms"] = e["terms"]` 줄 삭제 (또는 `e.get("terms", "")`) | 용어표 삭제로 `terms` 없음 → `KeyError`. 출력 파트(O9) GM 프롬프트에서 쓰는 값이라 배준호 님과 맞추기 | 없음 |


## 설정값

In [38]:
# 설정값

OLLAMA_URL = "http://localhost:11434"     # Ollama 서버 주소
EMBED_MODEL = "bge-m3"                    # 임베딩 모델 (글 → 숫자 벡터)
GEN_MODEL = "qwen3.5:9b"                 # 번역·선택·검증 모델
TRANSLATE_MODEL = GEN_MODEL               # 번역도 같은 모델 사용

REPO = "https://raw.githubusercontent.com/TRPG-masterbot/data/main/Min"
LOCAL = "/content"                               # 깃허브에서 받은 파일을 둘 곳 (colab에)
DATA = "/content/drive/MyDrive/KT 기본프로젝트"

OUT_PATH = f"{LOCAL}/rules_new.jsonl"         # 룰북 청크 파일
DEV_PATH = f"{LOCAL}/golden_dev2.json"        # 개발 질문 세트 149문항
FINAL_PATH = f"{DATA}/golden_final.json"      # 최종 질문 세트
EMB_PATH = f"{DATA}/rules_new_emb.npy"        # 임베딩 저장 파일
CACHE_PATH = f"{DATA}/cache_rules.json"       # 번역·검색 캐시

# ── 검색 ──
TOP_K = 5                                 # 최종으로 가져올 청크 수
POOL_K = 5                              # 검색 하나당 후보 수 (합치기 전)
SEARCH_WEIGHTS = [1, 1, 1, 0.5, 0.5]      # 검색 비중: 임베딩 3개, 키워드 2개
RRF_C = 60                                # RRF 점수 = 비중 / (RRF_C + 순위)
NUM_CTX = 8192                            # 문맥 길이

# ── 근거 고르기 ──
GEN_SEED = 42                             # 같은 질문 → 같은 결과
SELECT_CHARS = 300                        # 근거 고를 때 후보마다 보여 줄 글자 수
VERIFY_CHARS = 800                        # 근거 검증 때 보여 줄 글자 수
RULE_CHARS = 4000

# ── 동시 처리 ──
NUM_PARALLEL = 4                          # 모델 하나가 동시에 처리할 요청 수 (L4 기준 4: 14B 9GB + 대화 칸 1.6GB × 4)

## 0. 설치

In [2]:
%%writefile /content/requirements_rb.txt
pymupdf==2.0
requests==2.32.4
chromadb==1.5.9
langgraph==1.2.12

Writing /content/requirements_rb.txt


In [3]:
# 설치
!pip install -r /content/requirements_rb.txt -q
!pip freeze | grep -E "pymupdf|requests|chromadb|langgraph"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
chromadb==1.5.9
langgraph==1.2.12
langgraph-checkpoint==4.3.1
langgraph-prebuilt==1.1.1
langgraph-sdk==0.4.12
pymupdf==2.0
pymupdf4llm==2.0
pymupdf_layout==2.0
requests==2.32.4
requests-oauthlib==2.0.0
requests-toolbelt==1.0.0


## 1. Ollama 설치 · 서버 · 모델 받기
모델은 **백그라운드로** 받아요. 다운로드가 도는 동안 다음 셀들을 진행해도 돼요.

In [4]:
!apt-get install -y -qq zstd
!curl -fsSL http://ollama.com/install.sh | sh
!ollama --version

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
# import · 서버 켜기
import os, re, json, time, math, subprocess         # 파일·경로 / 정규식 / JSON / 시간 / 수학 / 명령 실행
from collections import Counter                     # 단어 개수 세기
import numpy as np, requests, chromadb              # 숫자 계산 / 웹 요청 / 벡터 DB
from tqdm.auto import tqdm                          # 진행 막대 (실행하면 보이는 그거)
from google.colab import drive, files               # 구글 드라이브 연결 / 파일 업·다운로드
from concurrent.futures import ThreadPoolExecutor   # 여러 작업 동시에 실행
from typing import TypedDict                        # 키가 정해진 딕셔너리 타입 (LangGraph 상태용)
from langgraph.graph import StateGraph, START, END  # 흐름 그래프 / 시작점 / 끝점

def wait(url, seconds=120):         # 서버가 켜질 때까지 1초마다 확인
    for _ in range(seconds):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print("서버 준비 완료:", url)
                return
        except Exception:
            pass                          # 아직 안 켜졌으면 에러 → 무시하고 다시
        time.sleep(1)
    print("서버가 아직 준비되지 않았습니다. ollama.log를 확인하세요")


os.environ["OLLAMA_KEEP_ALIVE"] = "1h"                                    # 모델을 1시간 동안 메모리에 유지
os.environ["OLLAMA_NUM_PARALLEL"] = str(NUM_PARALLEL)                     # 요청을 동시에 처리
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)    # 서버를 백그라운드로 켜기
wait(OLLAMA_URL)

서버 준비 완료: http://localhost:11434


In [6]:
# 모델 받기
subprocess.Popen(f'nohup sh -c "ollama pull {EMBED_MODEL} && ollama pull {GEN_MODEL}" > pull.log 2>&1 &', shell=True)
print("다운로드 시작. 진행 확인은 아래 셀")

다운로드 시작. 진행 확인은 아래 셀


In [7]:
# 모델 대기
t0 = time.time()
ready = False

while time.time() - t0 < 600:
    out = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout   # 받은 모델 목록
    if GEN_MODEL in out and EMBED_MODEL in out:
        print(f"모델 준비 완료 ({int(time.time() - t0)}초 기다림)")
        ready = True
        break
    print("아직 받는 중...", int(time.time() - t0), "초")
    time.sleep(15)

if not ready:
    print("10분이 지나도 안 받아졌어요. pull.log를 확인하세요:")
    print(open("pull.log").read()[-500:])

아직 받는 중... 0 초
아직 받는 중... 27 초
아직 받는 중... 42 초
아직 받는 중... 57 초
아직 받는 중... 73 초
아직 받는 중... 88 초
모델 준비 완료 (103초 기다림)


## 2. 구글 드라이브 · 경로

In [8]:
# 드라이브 연결
drive.mount("/content/drive")
print(OLLAMA_URL, "| LLM(번역·생성 공통):", GEN_MODEL, "| 임베딩:", EMBED_MODEL)

Mounted at /content/drive
http://localhost:11434 | LLM(번역·생성 공통): qwen3.5:9b | 임베딩: bge-m3


In [9]:
# 데이터 받기 (깃허브 → /content)
for name in ["rules_new.jsonl", "golden_dev2.json"]:
    res = requests.get(f"{REPO}/{name}", timeout=60)
    res.raise_for_status()                         # 파일 이름이나 주소가 틀리면 여기서 멈춤
    with open(f"{LOCAL}/{name}", "wb") as f:
        f.write(res.content)
    print("받음:", name, len(res.content), "bytes")

받음: rules_new.jsonl 1903588 bytes
받음: golden_dev2.json 19968 bytes


## 1. 청크 읽기 · 임베딩 · Chroma 컬렉션

In [10]:
# Ex01-a. 청크 읽기
rulechunks = []
for line in open(OUT_PATH, encoding="utf-8"):   # jsonl = 한 줄에 청크 하나
    c = json.loads(line)                        # 글자 -> 딕셔너리
    if c["meta"]["type"] == "rule":             # 규칙 청크만 검색 대상
        rulechunks.append(c)

print("검색 대상 청크:", len(rulechunks))            # 글꼴 크기(제목)로 자르되, 1500자 이상의 글은 나눠서

검색 대상 청크: 2180


In [11]:
# Ex01-b. 임베딩

def embed(texts, model=EMBED_MODEL):   # 글 여러 개 → 벡터 여러 개
    res = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": model, "input": texts, "options": {"num_ctx": 8192}},
        timeout=300,
    )
    res.raise_for_status()                       # 서버 오류면 여기서 멈춤
    return res.json()["embeddings"]


BATCH = 32

# 저장된 임베딩이 있고 청크 수도 같으면 → 다시 계산하지 않고 불러오기
saved = np.load(EMB_PATH) if os.path.exists(EMB_PATH) else None

if saved is not None and saved.shape[0] == len(rulechunks):
    emb = saved
    print("캐시 사용:", emb.shape)
else:
    vecs = []
    for i in tqdm(range(0, len(rulechunks), BATCH)):           #    32개씩 묶어서 반복
        batch = [c["text"] for c in rulechunks[i:i + BATCH]]   #    이번 묶음의 글만 뽑기
        vecs += embed(batch)                               #    서버에 요청해서 벡터를 이어 붙이기
    emb = np.array(vecs, dtype="float32")                  #    리스트 → 숫자 배열
    np.save(EMB_PATH, emb)
    print("새로 계산:", emb.shape)

# 캐시 사용: (2180, 1024)

캐시 사용: (2180, 1024)


In [12]:
# Ex01-c. Chroma 컬렉션
# 임베딩은 Ex01-b에서 만든 것을 그대로 넣음
client = chromadb.Client()                     # 메모리에만 만듦

# 같은 런타임에서 다시 실행해도 옛 벡터가 남지 않게, 있으면 지우고 새로 만듦
try:
    client.delete_collection("rules")
except Exception:
    pass                                       # 처음 실행이면 지울 게 없어서 에러 → 무시

rulebase = client.create_collection(
    name="rules",
    configuration={"hnsw": {"space": "cosine",    # 코사인 유사도로 비교
                            "ef_search": 200}},   # 검색할 때 더 넓게 살펴봄
)

ids = [str(i) for i in range(len(rulechunks))]                # Chroma id는 문자열: 청크 순서 번호를 그대로 사용
metadatas = [{"h1": c["meta"]["h1"]} for c in rulechunks]     # 장 이름
rulebase.add(ids=ids, embeddings=emb.tolist(), metadatas=metadatas)
print("컬렉션에 들어간 벡터 수:", rulebase.count())

# 컬렉션에 들어간 벡터 수: 2180

컬렉션에 들어간 벡터 수: 2180


## 2. 평가 도구 · 질문 세트

In [13]:
# Ex02. 제목
def norm(title):                                    # 제목 다듬기 (꼬리표 제거 + 소문자)
    title = re.sub(r"\s*\[[^\]]*\]$", "", title)    # "Prone [Condition]" → "Prone"
    title = re.sub(r"\s*\([^)]*\)$", "", title)     # "Hide (Action)"     → "Hide"
    title = title.replace("’", "'")                 # ’ → '
    title = re.sub(r"^Level \d+:\s*", "", title)    # 추가: "Level 5: Extra Attack" → "Extra Attack"
    return title.strip().lower()

def stem(title):                                    # 단수/복수 차이 없애기 (단어 끝 s 제거)
    return re.sub(r"s\b", "", norm(title))          # 예: "attacks" → "attack"

def hit(title, want):                               # 정답인지 (느슨한 비교: 단수/복수 차이 무시)
    return stem(title) == stem(want)

In [14]:
# Ex03. 채점

def wants(item):                                # 정답을 항상 리스트로
    want = item["want"]
    if want is None:
        return []
    if isinstance(want, str):                   # isinstance() : 값이 어떤 종류(타입)인지 확인
        return [want]
    return want

def rank(found, item):                          # 정답 등수 찾기 (없으면 None)
    titles = [c["meta"]["title"] for c in found]
    for i, title in enumerate(titles):          # enuemrate() : 리스트를 하나씩 꺼낼 때 번호도 같이 붙여줌
        if any(hit(title, w) for w in wants(item)):     # any() : 여러 값 중 하나라도 True면 True를 돌려줌
            return i + 1, titles
    return None, titles

def counts(ranks):                              # 순위 세기 → (1등, 3등 안, 5등 안)
    top1 = sum(1 for r in ranks if r == 1)
    top3 = sum(1 for r in ranks if r is not None and r <= 3)
    top5 = sum(1 for r in ranks if r is not None)
    return (top1, top3, top5)

def report(ranks):                              # 개수 · % · MRR(Mean Reciprocal Rank, 평균역순위) 한 줄로
    n = len(ranks)
    top1, top3, top5 = counts(ranks)
    mrr = sum(1 / r for r in ranks if r) / n    # 정답 순위의 역수 평균 (1등만 맞히면 1.0)
    print(f"1등 {top1} ({top1 / n:.1%}) | 3등 안 {top3} ({top3 / n:.1%}) | "
          f"5등 안 {top5} ({top5 / n:.1%}) | MRR {mrr:.3f} | {n}문항")

def ruleevaluate(search, items, misses=False):      # 질문 세트 전체 채점 (search = 검색 함수)
    ranks = []
    for g in tqdm(items, leave=False):
        found = search(g["q"])                  # g : 질문세트의 한 문항, q : 질문
        r, titles = rank(found, g)              # 정답이 몇등?
        ranks.append(r)
        if misses and r != 1:
            print(f"[{r}] {g['q']}\n     정답: {g['want']} | 검색: {titles[:3]}")
    report(ranks)
    return counts(ranks)

def judge(e, item):                 # evidence 결과 판정
    if item["want"] is None:        # 답이 없는 질문
        return "거절(맞음)" if e["status"] == "refuse" else "통과(틀림)"
    if e["status"] == "refuse":     # 답이 있는데 거절
        return "거절"
    if any(hit(e["title"], w) for w in wants(item)):    # 넘긴 규칙 제목이 정답이면
        return "정답 근거"
    return "다른 근거"

In [15]:
# Ex04. 질문 세트
def load(path):                     # 질문 세트 파일 읽기 (없으면 빈 리스트)
    if not os.path.exists(path):
        return []
    return json.load(open(path, encoding="utf-8"))

devset   = load(DEV_PATH)           # 개발 149문항 (정답 없는 거절 문항 포함)
finalset = load(FINAL_PATH)         # 최종 문항

# 정답 제목에 오타가 있으면 영원히 0점 → 룰북 제목 목록과 대조
known = {norm(c["meta"]["title"]) for c in rulechunks}

for name, items in {"devset": devset, "finalset": finalset}.items():    # 정답지에 적은 정답 제목이 실제 룰북 청크에 존재하는지 검사
    miss = []
    for g in items:                         # 문항을 하나씩
        for w in wants(g):                  # 그 문항의 정답 제목들을 하나씩
            if norm(w) not in known :
                miss.append(w)

In [ ]:
# answerable = [g for g in devset if g["want"] is not None]
# print("devset:", len(devset), "문항 | 답 있음", len(answerable), "| 답 없음", len(devset) - len(answerable))

## 4. 번역 (한국어 질문 → 영어 검색어)
- 번역①: 일반 번역 (예시 3개)
- 번역②: 답이 될 룰북 항목 이름 최대 3개
- 호출은 **9B, temperature 0, seed 42, num_ctx 8192**

In [17]:
# Ex06. 번역 호출

def ask(prompt, model=None, temperature=0.0):      # 번역용 LLM 호출
    options = {
        "temperature": temperature,                # 0
        "seed": GEN_SEED,
        "num_ctx": NUM_CTX,
    }
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or TRANSLATE_MODEL,       # model을 안 주면 기본 번역 모델
              "messages": [{"role": "user", "content": prompt}],
              "stream": False,
              "think": False,                          # 생각 모드 끄기 (3.5용)
              "options": options},
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

In [40]:
# Ex07. 번역 프롬프트

def clean(text):                    # 번역 결과 다듬기 (첫 줄만, "Query:"·따옴표·영어 아닌 글자 제거)
    text = text.strip().split("\n")[0]                        # 첫 줄만
    text = re.sub(r"^\s*query\s*:\s*", "", text, flags=re.I)  # 앞의 "Query:" 지우기 (대소문자 무시)
    text = re.sub(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]+", " ", text)   # 한자·일본어·한글 → 공백 (예: "Poison中毒效果" → "Poison")
    text = re.sub(r"\s+", " ", text)                          # 공백 여러 개 → 하나
    return text.strip().strip("\"'")                          # 양 끝 따옴표 지우기


# ── 번역① : 일반 번역 (예시 3개를 보여 주고 따라 하게) ──
REWRITE_TEMPLATE = """Convert the user's question about D&D 5.2 rules into a short English search query.
Use the official rule terms that the rulebook would use.
Output only the query, nothing else.

Question: 숙련 보너스는 레벨마다 얼마야?
Query: Proficiency Bonus by character level

Question: 강을 헤엄쳐서 건너도 돼?
Query: Swimming rules
# 한글로@@
Question: 회피의 반지가 뭐야?
Query: Ring of Evasion

Question: {question}
Query:"""


def translate1(question):           # 번역① 일반 번역
    prompt = REWRITE_TEMPLATE.format(question=question)     # format() : 문자열 안의 {} 빈칸에 값을 끼워 넣는 함수
    return clean(ask(prompt))
# fewshot

# ── 번역② : 답이 될 룰북 항목 이름 최대 3개 ──
REWRITE2 = """Name the D&D 5.2 rulebook entries that could answer the user's question:
the rule, action, condition, spell, class feature, item, or creature, written as its heading in the rulebook.
Give up to 3 names on one line, separated by commas, most likely first. Output only the names, nothing else.

Question: {q}
Names:"""


def translate2(question):
    prompt = REWRITE2.format(q=question)
    return clean(ask(prompt))

## 5. 키워드 검색 BM25
임베딩은 **뜻**, BM25는 **글자 그대로의 단어**를 비교

In [19]:
# Ex08. 키워드 검색

# 불용어제거 : 너무 흔해서 검색에 도움이 안 되는 단어 제거
STOP = set("a an the of to in on and or is are be can you your it its as at by for with that this if when how what do does".split())


def tokenize(text):                 # 글 → 단어 목록 (소문자, 영어·숫자만, 불용어 제외)
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOP]


class BM25:
    # 점수가 높아지는 경우: 단어가 많이 나옴(tf) + 드문 단어(idf) + 짧은 청크(len)
# k1 : 단어가 많이 나올 때 점수가 얼마나 오를지, b : 긴 청크를 얼마나 깎을지
    def __init__(self, docs, k1=1.5, b=0.75):      # 청크들의 단어 통계 미리 계산
        self.k1 = k1
        self.b = b
        self.tf = [Counter(d) for d in docs]       # 청크마다 단어별 등장 횟수
        self.len = np.array([len(d) for d in docs], dtype="float32")   # 청크마다 단어 수
        self.avg = self.len.mean()                 # 평균 단어 수

        # df = 그 단어가 나오는 청크 수 (한 청크에서 여러 번 나와도 1번으로)
        df = Counter()
        for d in docs:
            for w in set(d):                       # set() : 중복제거
                df[w] += 1

        # idf = 드문 단어일수록 큰 값
        total = len(docs)                          # 전체 청크 수
        self.idf = {}
        for w, n in df.items():
            self.idf[w] = math.log(1 + (total - n + 0.5) / (n + 0.5))

    def scores(self, query):                       # 질문 → 청크마다 점수
        s = np.zeros(len(self.tf), dtype="float32")# 청크마다 점수 0으로 시작
        for w in set(tokenize(query)):             # 질문의 단어를 하나씩 (중복 제거- set())
            if w not in self.idf:                  # 룰북에 없는 단어는 건너뜀
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(w, 0)                   # 이 청크에서 단어가 나온 횟수
                if f:
                    scale = 1 - self.b + self.b * self.len[i] / self.avg   # 길이 보정 (긴 청크일수록 큼)
                    s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * scale)
                    # 희귀도 × 빈도 효과 / 길이 보정
        return s


# 제목을 두 번 넣어서 제목 단어에 가중치 -> 제목이 점수를 더 받게 하려구
docs = []
for c in rulechunks:
    title = c["meta"]["title"]      # 딕셔너리 안의 딕셔너리에서 값을 꺼내는 방식
    docs.append(tokenize(title + " " + title + " " + c["text"]))
bm25 = BM25(docs)
# 왜쓰는지 사용법

def keyword(question, k=10):        # 키워드 검색 → 점수 높은 청크 k개
    order = np.argsort(-bm25.scores(question))            # 점수 큰 순서의 번호 (-를 붙여 내림차순)
    return [rulechunks[i] for i in order[:k]]

## 6. 최종 검색 `retrieve` (임베딩 3갈래 + 키워드 2갈래 → 가중 RRF)

In [20]:
# Ex09. 최종 검색

def semantic(texts, k=5):           # 임베딩 검색 (뜻으로 찾기: 글 여러 개 → 비슷한 청크 k개)
    vectors = embed(texts)
    res = rulebase.query(query_embeddings=vectors, n_results=k)

    # 같은 청크가 여러 번 나오면 가장 높은 점수만 남김
    best = {}
    for ids, dists in zip(res["ids"], res["distances"]):     # 글마다 결과 k개
        for key, dist in zip(ids, dists):                    # zip() : 여러 리스트를 같은 순서끼리 짝지어 한 쌍씩 꺼내 주는 함수
            i = int(key)                        # Chroma id = 청크 순서 번호 (문자열)
            score = 1 - dist                    # 코사인 거리 → 유사도 (클수록 비슷)
            best[i] = max(best.get(i, -1.0), score)

    order = sorted(best, key=best.get, reverse=True)   # 점수 높은 순
    return [rulechunks[i] for i in order[:k]]

def semantic2(texts, k=5):       # 임베딩 검색을 글마다 따로 (임베딩 호출은 한 번에)
    vectors = embed(texts)       # 글 여러 개를 한 번의 호출로 벡터로
    res = rulebase.query(query_embeddings=vectors, n_results=k)

    lists = []
    for ids in res["ids"]:                  # 글마다 결과 k개 (가까운 순)
        found = []
        for key in ids:
            found.append(rulechunks[int(key)])  # Chroma id = 청크 순서 번호
        lists.append(found)
    return lists

def merge(lists, weights, k=TOP_K, c0=RRF_C):     # 여러 검색 결과를 순위 점수(RRF)로 합치기
    # 점수 = 비중 / (c0 + 순위) → 여러 목록에서 위에 있을수록 점수가 높아짐
    score, lookup = {}, {}
    for items, w in zip(lists, weights):
        for r, c in enumerate(items, 1):           # r = 순위 (1부터)
            score[c["id"]] = score.get(c["id"], 0) + w / (c0 + r)
            lookup[c["id"]] = c                    # id → 청크 (마지막에 되찾으려고)

    order = sorted(score, key=score.get, reverse=True)      # sorted() : 값들을 정렬한 새 리스트를 돌려줌
    return [lookup[i] for i in order[:k]]

pool = ThreadPoolExecutor(max_workers=NUM_PARALLEL) # 여러 작업을 동시에 실행해 주는 도구
# 공부@
def retrieve(question, k=TOP_K):                        # 최종 검색
    job1 = pool.submit(translate1, question)            # 번역1
    job2 = pool.submit(translate2, question)            # 번역2
    job0 = pool.submit(semantic, [question], POOL_K)    # 임베딩 : 원문 질문

    # 끝날 때까지 기다렸다가 결과 받기
    en1 = job1.result()
    en2 = job2.result()
    first = job0.result()

    pair = semantic2([en1, en2], POOL_K)             # 임베딩: 번역1, 2를 한 번에

    lists = [
        first,                                       # 임베딩: 원문 질문
        pair[0],                                     # 임베딩: 번역1
        pair[1],                                     # 임베딩: 번역2
        keyword(en1, POOL_K),                        # 키워드: 번역1
        keyword(en2, POOL_K),                        # 키워드: 번역2
    ]
    return merge(lists, SEARCH_WEIGHTS, k), [question, en1, en2]    # (청크 5개, 검색에 쓴 글 3개)

## 7. 번역·검색 캐시 (개발용, 꺼 둠)
청크나 검색 방식을 바꾸는 실험 중에는 옛 결과가 나올 수 있어서 꺼 뒀어요.

In [21]:
# Ex10-a. 캐시 장치 (꺼 둠)
# 같은 질문을 다시 물으면 모델을 부르지 않고 저장해 둔 결과를 바로 돌려줌. -> 개발/실험용

# by_id = {c["id"]: c for c in rulechunks}     # 청크 id → 청크 (파일에서 불러올 때 id로 청크를 되찾으려고)

# _tr_cache = {}   # 번역 기억장:  ("1", 질문) 또는 ("2", 질문)  →  영어 검색어
# _sf_cache = {}   # 검색 기억장:  (질문, 개수)                 →  (청크 목록, [질문, 번역①, 번역②])


# # ── 1. 원래 함수 보관 ─────────────────────────────────────────
# # 셀을 두 번 돌려도 "캐시 함수를 또 캐시로 감싸는" 일이 없게, 처음 한 번만 보관.
# if "_orig_tr1" not in globals():
#     _orig_tr1 = translate1
#     _orig_tr2 = translate2
#     _orig_retrieve = retrieve


# # ── 2. 같은 이름으로 캐시 버전 만들기 ─────────────────────────
# # 이름이 같아서, 다른 셀(Ex07 retrieve 안의 번역 호출 등)도 자동으로 캐시 버전을 써요.
# def translate1(q):                          # 번역① (캐시 버전)
#     key = ("1", q)
#     if key not in _tr_cache:              # 처음 보는 질문일 때만 모델 호출
#         _tr_cache[key] = _orig_tr1(q)
#     return _tr_cache[key]


# def translate2(q):                          # 번역② (캐시 버전)
#     key = ("2", q)
#     if key not in _tr_cache:
#         _tr_cache[key] = _orig_tr2(q)
#     return _tr_cache[key]


# def retrieve(q, k=TOP_K):                  # 검색 (캐시 버전)
#     key = (q, k)
#     if key not in _sf_cache:
#         _sf_cache[key] = _orig_retrieve(q, k)
#     return _sf_cache[key]


# # ── 3. 드라이브에 저장 / 불러오기 ─────────────────────────────
# # JSON은 튜플 키를 못 쓰니까 "1|질문", "질문|5"처럼 |로 이어 붙인 글자로 저장해요.
# # 청크는 통째로 저장하지 않고 id만 저장해요 (불러올 때 by_id로 되찾음).
# def save():                                 # 캐시를 드라이브에 저장
#     tr = {}
#     for (kind, q), en in _tr_cache.items():
#         tr[f"{kind}|{q}"] = en

#     sf = {}
#     for (q, k), (found, texts) in _sf_cache.items():
#         sf[f"{q}|{k}"] = {"ids": [c["id"] for c in found], "texts": texts}

#     with open(CACHE_PATH, "w", encoding="utf-8") as f:
#         json.dump({"tr": tr, "sf": sf}, f, ensure_ascii=False)
#     print("저장:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")


# def restore():                              # 드라이브에서 캐시 불러오기
#     if not os.path.exists(CACHE_PATH):
#         print("캐시 파일 없음 → 아래 build(dev)로 만들어요")
#         return

#     with open(CACHE_PATH, encoding="utf-8") as f:
#         d = json.load(f)

#     for key, en in d["tr"].items():
#         kind, q = key.split("|", 1)            # 앞에서 한 번만 자름: "1|질문"
#         _tr_cache[(kind, q)] = en

#     for key, v in d["sf"].items():
#         q, k = key.rsplit("|", 1)              # 뒤에서 한 번만 자름: "질문|5" (질문에 |가 있어도 안전)
#         found = [by_id[i] for i in v["ids"]]
#         _sf_cache[(q, int(k))] = (found, v["texts"])

#     print("불러옴:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")


# # ── 4. 모델 내리기 ───────────────────────────────────────────
# # keep_alive 0 = 지금 바로 GPU 메모리에서 내림 (14B ↔ bge-m3 교체를 줄이려고)
# def unload(model):                          # 모델을 GPU 메모리에서 내리기
#     requests.post(f"{OLLAMA_URL}/api/generate", json={"model": model, "keep_alive": 0})


# # ── 실행 ────────────────────────────────────────────────────
# dev = devset
# restore()

# 불러옴: 132 번역 / 66 검색

In [ ]:
# Ex10-b. 캐시 만들기 (꺼 둠)

# def build(items):             # 질문 세트의 번역·검색 결과를 미리 만들어 두기
#     qs = [g["q"] for g in items]

#     # 1. 번역: 번역①② 중 하나라도 없는 질문만
#     todo_tr = []
#     for q in qs:
#         if ("1", q) not in _tr_cache or ("2", q) not in _tr_cache:
#             todo_tr.append(q)

#     if todo_tr:
#         _orig_tr1(todo_tr[0])                      # 워밍업: 14B를 메모리에 올리기 (결과는 버림)
#         for q in tqdm(todo_tr, desc="번역(14B)"):
#             translate1(q)
#             translate2(q)
#         unload(TRANSLATE_MODEL)                    # 14B 내리기 → 임베딩 모델 자리 비우기

#     # 2. 검색: 검색 결과가 없는 질문만
#     todo_sf = []
#     for q in qs:
#         if (q, TOP_K) not in _sf_cache:
#             todo_sf.append(q)

#     for q in tqdm(todo_sf, desc="검색(bge-m3)"):
#         retrieve(q)
#     if todo_sf:
#         unload(EMBED_MODEL)

#     print(f"완료. 번역 새로 {len(todo_tr)}개, 검색 새로 {len(todo_sf)}개 → 캐시 번역 {len(_tr_cache)} / 검색 {len(_sf_cache)}")


# build(dev)
# save()

# 검색(bge-m3): 0/0 [00:00<?, ?it/s]
# 완료. 번역 새로 0개, 검색 새로 0개 → 캐시 번역 132 / 검색 66
# 저장: 132 번역 / 66 검색

## 8. LLM 호출 (근거 선택·검증용 짧은 답)

In [22]:
# Ex11. LLM 호출

# 시스템 프롬프트
SYSTEM = ("You are a D&D rules assistant.")

def chat(prompt, model=None, system=SYSTEM, temperature=0.0, seed=GEN_SEED, limit=None):
    options = {"num_ctx": NUM_CTX, "temperature": temperature}
    if seed is not None:
        options["seed"] = seed                     # seed가 있으면 같은 결과로 고정
    if limit:
        options["num_predict"] = limit             # 출력 길이 제한 (짧은 답용)

    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or GEN_MODEL,
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": prompt}],
              "stream": False,
              "think": False,
              "options": options},
        timeout=600,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]


def ask3(prompt, model=None, system=SYSTEM):         # 짧은 답용 호출 (번호, 예/아니오) — 최대 20토큰
    return chat(prompt, model, system, 0.0, GEN_SEED, limit=20)

## 9. 근거 파이프라인 (LangGraph)

```
START → search(번역①② 동시 + 검색) → choose(근거 번호) → check(검증) ┬ 통과 → accept(원본 전체 근거) → END
                                                                    └ 실패 → refuse(거절) → END
```
- check: 고른 근거를 먼저 검증 → 실패하면 나머지 4개를 동시에 검증 → 검색 순위가 높은 통과 후보
- 한국어 설명은 GM이 써요

In [29]:
# Ex12. 근거 선택
SELECT_TEMPLATE = """다음은 D&D 규칙 질문과 후보 근거입니다. 질문에 직접 답하는 근거의 번호 하나만 고르세요.
- 질문이 묻는 규칙 자체를 설명하는 근거를 고르세요. 이름만 비슷하거나 예시로 지나가듯 언급되는 근거는 고르지 마세요.
- 숫자 하나만 출력하세요.

### 질문
{question}

### 후보 근거
{cands}

### 번호"""


def select(question, found, model=None):   # 후보 중 질문에 맞는 근거 번호 고르기 (1부터)
    # 후보마다 "[번호] 제목: 본문 앞부분 300자" 한 줄씩
    lines = []
    for n, c in enumerate(found, 1):
        preview = c["text"][:SELECT_CHARS].replace("\n", " ")   # 줄바꿈 → 공백 (한 줄로)
        lines.append(f"[{n}] {c['meta']['title']}: {preview}")
    cands = "\n".join(lines)

    out = ask3(SELECT_TEMPLATE.format(question=question, cands=cands), model)

    # 답에서 첫 숫자 꺼내기. 숫자가 없거나 범위 밖이면 1번 -> llm답이 이상하면 무시하고 1등을 믿을게
    m = re.search(r"\d+", out)
    n = int(m.group()) if m else 1
    if 1 <= n <= len(found):
        return n
    return 1

# def prefer(found, texts):           # ⑧-2 번역②(제목 맞히기)와 제목이 정확히 같은 후보 번호
#     name = norm(texts[2])
#     for i, c in enumerate(found, 1):
#         if norm(c["meta"]["title"]) == name:
#             return i
#     return None

In [24]:
# Ex13. 원본 전체 읽기
# 검색은 나뉜 조각으로, GM에게 넘길 근거는 원본 전체로

def heading(chunk):                 # 청크 머리줄 (장 > 절 > 제목)
    return chunk["text"].split("\n", 1)[0]


def content(chunk):                 # 청크 본문 (둘째 줄부터)
    return chunk["text"].split("\n", 1)[1]


# 나뉜 조각끼리 모아 두기: (제목, 쪽) → {조각 번호: 청크}
parts = {}
for c in rulechunks:
    meta = c["meta"]
    if "part" not in meta:                      # 나뉘지 않은 청크는 건너뜀
        continue
    key = (meta["title"], meta["page"])
    if key not in parts:
        parts[key] = {}
    parts[key][meta["part"]] = c


def expand(chunk):                  # 나뉜 조각이면 원본 전체(모든 조각)를 이어서 돌려줌
    meta = chunk["meta"]
    if "part" not in meta:                      # 나뉘지 않은 청크는 그대로
        return chunk

    group = parts[(meta["title"], meta["page"])]

    texts = []
    for k in range(1, meta["parts"] + 1):       # 1번 조각부터 마지막 조각까지 순서대로(k = i(번호), x(문자열, 청크 등))
        texts.append(content(group[k]))
    whole = " ".join(texts)

    return {"id": chunk["id"], "text": heading(chunk) + "\n" + whole, "meta": meta}

In [42]:
# Ex14. 근거 검증
VERIFY_TEMPLATE = """다음 D&D 규칙 질문과 근거입니다. 이 근거가 질문에 직접 답하는 규칙 내용을 담고 있으면 '예', 이름만 비슷하거나 다른 주제이면 '아니오'라고만 답하세요.
질문은 한국어, 근거는 영어예요. 표현이 달라도 같은 규칙을 다루면 '예'예요.
주제만 비슷하거나 관련 내용이 조금 있는 정도면 "아니오"입니다. 질문에 직접 답하는 문장이 있을 때만 "예"라고 답하세요.

### 질문
{question}

### 근거
{title}: {text}

### 답 (예/아니오)"""

def verify(question, chunk, limit=VERIFY_CHARS):                 # 근거가 질문에 맞는지 LLM에게 묻기 (예 → True)
    prompt = VERIFY_TEMPLATE.format(question=question, title=chunk["meta"]["title"],
                                    text=chunk["text"][:limit])
    out = ask3(prompt)
    out = re.sub(r"[\s\*'\"]", "", out)               # 공백·별표·따옴표 지우기 ("**예**" → "예")
    return out.startswith(("예", "네", "yes", "Yes"))

def match(texts, chunk):            # 근거 제목이 번역① 안에 그대로 있는지
    t = norm(chunk["meta"]["title"])
    if re.search(r"\b" + re.escape(t) + r"\b", texts[1].lower()):
        return True
    return False

def verify2(question, chunk, texts, limit=VERIFY_CHARS):    # 근거 통과 여부 (제목이 맞으면 바로 통과, 아니면 LLM 검증)
    return match(texts, chunk) or verify(question, chunk, limit)

def place(chunk):
    return (chunk["meta"]["title"], chunk["meta"]["page"])

def pick(question, found, texts, n):    # 고른 근거부터 검증해서 처음 통과한 번호 (모두 실패면 None)
    seen = set()                        # 이미 검증한 규칙
    # 1단계 : 고른 번호가 범위 안이면 그 규칙(원본 전체)부터 검증
    if n is not None and 1 <= n <= len(found):
        seen.add(place(found[n-1]))
        if verify2(question, expand(found[n - 1]), texts, RULE_CHARS):
            return n

    # 2단계 : 아직 안 본 규칙만 모아서 동시에 검증 (같은 규칙의 다른 조각은 건너뜀)
    others = []
    for i in range(1, len(found) + 1):
        p = place(found[i - 1])
        if p in seen:
            continue
        seen.add(p)
        others.append(i)

    jobs = []
    for i in others:
        jobs.append(pool.submit(verify, question, expand(found[i - 1]), RULE_CHARS))

    for i, job in zip(others, jobs):        # 순위가 높은 후보부터 → 처음 통과한 번호
        if job.result():
            return i
    return None

In [34]:
# Ex15. 최종 함수 (LangGraph)
# 질문 → 검색 → 근거 선택 → 검증 → 근거 or 거절

class State(TypedDict, total=False):    # 노드끼리 주고받는 값
    question: str                       # 질문
    found: list                         # 검색 후보 5개
    texts: list                         # [원문, 번역1, 번역2]
    n: int                              # 고른 번호 → 검증 후 통과 (모두 실패면 None)
    result: dict                        # 최종 결과

def rulesearch(state):      # 노드: 룰북 검색
    found, texts = retrieve(state["question"])
    return {"found": found, "texts": texts}

def choose(state):          # 노드: 후보 중 근거 번호 고르기
    n = select(state["question"], state["found"])
    return {"n": n}

def check(state):           # 노드: 고른 근거부터 검증 (실패하면 나머지 동시 검증)
    n = pick(state["question"], state["found"], state["texts"], state["n"])
    return {"n": n}

def route(state):           # 갈림길: 통과 → accept, 모두 실패 → refuse
    if state["n"] is None:
        return "refuse"
    return "accept"

def accept(state):          # 노드: 검증된 근거를 GM에게 넘길 모양으로
    c = expand(state["found"][state["n"] - 1])     # 나뉜 청크면 원본 전체
    result = {
        "status": "ok",
        "title": c["meta"]["title"],
        "page": c["meta"]["page"],
        "path": heading(c),                        # 장 > 절 > 제목 <-어디?
        "text": content(c),                        # 영어 본문 (GM 프롬프트에 넣을 근거)
        "picked": state["n"],                      # 후보 중 몇 번째였는지
        "queries": state["texts"][1:],             # 번역1, 2
    }
    return {"result": result}

def refuse(state):
    candidates = []
    for x in state["found"][:3]:
        candidates.append((x["meta"]["title"], x["meta"]["page"]))
    result = {
        "status": "refuse",
        "candidates": candidates,
        "queries": state["texts"][1:],
    }
    return {"result": result}

# 그래프 만들기 노드등록 -> 연결 -> 컴파일
graph = StateGraph(State)
graph.add_node("search", rulesearch)   # 노드 이름은 "search" 그대로
graph.add_node("choose", choose)
graph.add_node("check", check)
graph.add_node("accept", accept)
graph.add_node("refuse", refuse)

graph.add_edge(START, "search")
graph.add_edge("search", "choose")
graph.add_edge("choose", "check")
graph.add_conditional_edges("check", route, ["accept", "refuse"])   # 검증 결과에 따라 갈림
graph.add_edge("accept", END)
graph.add_edge("refuse", END)

app = graph.compile()

def evidence(question):         # 질문에 답하는 룰북 근거 찾기 (GM이 부르는 함수)
    state = app.invoke({"question": question})
    return state["result"]

In [27]:
# Ex16. 화면 출력

def show(question):                 # 근거를 화면에 출력 (개발 확인용)
    e = evidence(question)

    if e["status"] == "refuse":
        print("답을 못찾았어요.")
        print(" 후보:", e["candidates"], "   번역:", e["queries"])
        return e

    print(f"출처: {e['title']} p.{e['page']} ({e['path']})")
    print("근거:", e["text"][:600])
    return e

### 질문해 보기

In [ ]:
# question = "넘어지면 어떤 문제가 발생해?"
# _ = show(question)

## 📝 실험 기록

**검색**
- 번역을 7B → 14B로 바꿔 개발셋 검색을 다시 쟀다 → 66문항 합계 (43, 52, 59)로 7B (41, 54, 57)와 비슷.
- 번역 방식 A~E 비교 → 세 갈래 RRF(E)가 가장 좋아서 채택.
- 키워드(BM25) 합치기 비교 → 키워드 비중 0.5(F2)를 검색 v1로 채택.
- 번역①의 용어 힌트를 빼 봤다 → 점수가 비슷해서 힌트 없는 일반 번역으로 확정.
- 한국어 + 번역 두 번 검색해서 합치기 → 이득 없어서 기각.
- 재순위(reranker) 실험 → 1등이 오히려 줄어서(31 → 28) 기각. -> 재시도
- Prone·Hide 질문은 어떤 방식으로도 잘 못 찾음 → 알려진 약점. -> 더 있을지도?

**답변**
- 근거 선택을 "한 번에 하기" vs "2단계(번호 먼저)"로 비교 → 2단계 채택.
- 근거 검증(예/아니오) 평가 + 거절 정책 비교 → 정책 A(아니오면 바로 거절) 채택.
- 설명 충실성 검사(근거에 없는 내용 묻기)를 추가했다.
- 7B·14B 설명에서 한자 섞임, 분수 뒤집힘(3분의 4) 발견 → 깨짐·혼합·분수 검사와 재생성으로 이어짐.
- 용어표 주입 켬/끔 비교(38문항) → 켬: 정상 31·숨김 3·용어누락 9 / 끔: 정상 28·숨김 6·용어누락 18 → 켬 유지.
- 모델 교체(14B ↔ bge-m3)가 느림의 원인 → 번역·검색 캐시를 만들었다.
- temperature 0 + seed 42로 같은 결과가 나오는지 확인 → 재현됨.

**최종 세트 (golden_final, 60문항)**
- v1 점수는 31 / 46 / 51. 한 번만 재는 시험지라 결과 파일이 있으면 다시 못 재게 잠가 뒀었다.

**2026-11-12 변경 (개발 66문항 기준)**
- FAISS → Chroma(코사인) + 하이픈 살리기 → 검색 (43, 52, 59) → (45, 54, 58)
- 긴 청크 1500자 나누기: 검색 1등 ±1 (잡음 범위), 2000자 넘던 87개 청크를 답변 단계에서 읽을 수 있게 됨
- 나눌 때 겹치기 2문장 → 1등 46 → 44로 오히려 하락 → 기각. 대신 답변 때 원본 전체를 읽게 함(`expand`, NUM_CTX 8192)
- 표 → 문장 변환: PDF에서 띄어쓰기·열이 깨져서 기각
- 모델 비교 (정상/숨김/거절): qwen2.5 14B 55/6/5, qwen3.5 9B 49/8/9, qwen2.5 7B 31/?/17 → 14B 유지
- 캐시는 실험 중 꺼 둠 (청크·검색이 바뀌면 옛 결과가 나오는 문제)

**2026-11-13 변경 (개발 149문항 기준, 정답 있는 136문항)**
- 개발셋 66 → 149문항 (거절 문항 13개 포함)
- 검색 기준: (87, 113, 119)
- `pick`(고른 근거가 아니면 다음 후보도 검증): 정상 101 → 115, 거절 25 → 10, 지어냄 3 그대로 → 채택
- `squash`(띄어쓰기·하이픈 무시 원문 검증): 효과 없음 → 원인이 띄어쓰기가 아니었음
- 용어표 v2: 일반 단어 38개 제거, 상태이상·행동·클래스 기능·마스터리·재주 추가 (200개), 번역② 27개는 용어표에서 뽑음
- 설명 파트(설명 쓰기·검사) 제거 → 룰 RAG는 검증된 근거만 넘기고 설명은 GM이 씀 (질문당 시간: 설명 쓰기 약 3.6초 빠짐)

**2026-11-13 변경 2 (개발 149문항: 정답 있는 137 / 거절 12)**
- 질문 세트 보정: 상위 절 근거 5개를 정답에 추가, 바드 문항 → 정답 있는 문항, 숲 문항 표현 수정, Grappl → Grappled
- 채점: hit2(단수/복수 무시)로 통일, ALIAS(특정 값 예외표) 제거
- 데이터: 깃허브 TRPG-masterbot/data/Min에서 받음 (임베딩 파일만 드라이브)
- 기준값: 검색 (91, 117, 122) / 근거 정답 114 · 다른 근거 13 · 거절 10 / 거절 문항 거절(맞음) 10 · 통과(틀림) 2
- 동시 실행(NUM_PARALLEL 4): 번역①②·원문 임베딩 동시, 검증 2단계(실패 시 나머지 동시) → 결과 그대로, 질문당 약 2초 → 1.7초
- LangGraph로 전환(search → choose → check → accept/refuse) → 결과 그대로
- v1.5 수정: Ex09 임베딩 중복 호출 제거, pick 2단계에서 실패한 번호 재검증 제거, 번역에 섞인 한자 제거(clean), Ex02 return 누락 수정
- v1.6 수정: 실험1·2 통합(검색 한 번으로 채점), 제목 비교 통일(따옴표 ’→', "Level N:" 제거), 검증에서 영어 검색어 빼기, 결합 대비 이름 변경(chunks→rulechunks, collection→rulebase, search 노드 함수→rulesearch, evaluate→ruleevaluate)


[17. 평가 통합 — 14B]
1등 92 (67.2%) | 3등 안 116 (84.7%) | 5등 안 122 (89.1%) | MRR 0.762 | 137문항
[답 있는 질문] 정답 근거 113 (82.5%) / 다른 근거 12 / 거절 12
[답 없는 질문] 거절(맞음) 10 / 통과(틀림) 2
질문당 1.3 초

[18-1. 따옴표 통일 (match 미적용) — 14B]
1등 93 (67.9%) | 3등 안 116 (84.7%) | 5등 안 122 (89.1%) | MRR 0.767 | 137문항
[답 있는 질문] 정답 근거 112 (81.8%) / 다른 근거 13 / 거절 12
[답 없는 질문] 거절(맞음) 10 / 통과(틀림) 2
질문당 1.4 초

[18. 제목 통일 (따옴표 + Level) — 14B]
1등 92 (67.2%) | 3등 안 117 (85.4%) | 5등 안 123 (89.8%) | MRR 0.765 | 137문항
[답 있는 질문] 정답 근거 115 (83.9%) / 다른 근거 10 / 거절 12
[답 없는 질문] 거절(맞음) 10 / 통과(틀림) 2
질문당 1.3 초

[19. 검증 영어 빼기 — 14B]
1등 92 (67.2%) | 3등 안 116 (84.7%) | 5등 안 123 (89.8%) | MRR 0.765 | 137문항
[답 있는 질문] 정답 근거 117 (85.4%) / 다른 근거 12 / 거절 8
[답 없는 질문] 거절(맞음) 10 / 통과(틀림) 2
질문당 1.3 초

[20. 9B 전환 + pick ① 번호 범위 확인]
1등 90 (65.7%) | 3등 안 115 (83.9%) | 5등 안 122 (89.1%) | MRR 0.752 | 137문항
[답 있는 질문] 정답 근거 106 (77.4%) / 다른 근거 24 / 거절 7
[답 없는 질문] 거절(맞음) 9 / 통과(틀림) 3
질문당 1.8 초

[21. 9B + pick ② 같은 규칙 한 번만, 원본 전체로 검증]
1등 91 (66.4%) | 3등 안 116 (84.7%) | 5등 안 123 (89.8%) | MRR 0.759 | 137문항
[답 있는 질문] 정답 근거 107 (78.1%) / 다른 근거 24 / 거절 6
[답 없는 질문] 거절(맞음) 9 / 통과(틀림) 3
질문당 1.1 초

[22. 9B + pick ③ 여러 개 통과 시 다시 고르기]
1등 91 (66.4%) | 3등 안 116 (84.7%) | 5등 안 123 (89.8%) | MRR 0.759 | 137문항
[답 있는 질문] 정답 근거 107 (78.1%) / 다른 근거 24 / 거절 6
[답 없는 질문] 거절(맞음) 8 / 통과(틀림) 4
질문당 1.1 초
1 (9B + ①②, 채택)	91 / 116 / 123	0.759	107 / 24 / 6	9 / 3	1.1초
23 (9B + ②, ① 끔)	89 / 116 / 123	0.752	107 / 24 / 6	9 / 3	1.2초

In [43]:
# 실험1. 검색 · 근거 · 시간
ask3("안녕")                                   # 워밍업: 모델을 먼저 올려 둠
normal = Counter()                             # 답 있는 질문: 정답 근거 / 다른 근거 / 거절
refusal = Counter()                            # 답 없는 질문: 거절(맞음) / 통과(틀림)
bytype = {}                                    # 질문 종류별 판정
problems = []                                  # 직접 볼 문항
ranks = []                                     # 검색 순위 (답 있는 질문만)

t0 = time.time()
for g in tqdm(devset):
    state = app.invoke({"question": g["q"]})   # evidence 대신 상자 전체 받기
    e = state["result"]
    result = judge(e, g)

    if g["want"] is None:                      # 답 없는 질문은 검색 채점에서 뺌
        refusal[result] += 1
    else:
        normal[result] += 1
        r, titles = rank(state["found"], g)    # 같은 검색 결과로 순위 채점
        ranks.append(r)

    kind = g.get("type", "기존")
    if kind not in bytype:
        bytype[kind] = Counter()
    bytype[kind][result] += 1

    if result not in ("정답 근거", "거절(맞음)"):
        problems.append((result, g["q"], g["want"], e.get("title") or e.get("candidates")))

print("[검색]")
report(ranks)                                  # 1등 · 3등 안 · 5등 안 · MRR

n = sum(normal.values())
print(f"\n[답 있는 질문] 정답 근거 {normal['정답 근거']} ({normal['정답 근거'] / n:.1%}) / "
      f"다른 근거 {normal['다른 근거']} / 거절 {normal['거절']}")
print(f"[답 없는 질문] 거절(맞음) {refusal['거절(맞음)']} / 통과(틀림) {refusal['통과(틀림)']}")
print(f"질문당 {(time.time() - t0) / len(devset):.2f}초")

print("\n[종류별]")
for kind, c in bytype.items():
    print(kind, dict(c))

print("\n[직접 볼 문항]")
for p in problems:
    print(p)

  0%|          | 0/149 [00:00<?, ?it/s]

[검색]
1등 98 (71.5%) | 3등 안 120 (87.6%) | 5등 안 127 (92.7%) | MRR 0.800 | 137문항

[답 있는 질문] 정답 근거 112 (81.8%) / 다른 근거 12 / 거절 13
[답 없는 질문] 거절(맞음) 10 / 통과(틀림) 2
질문당 1.23초

[종류별]
기존 {'정답 근거': 52, '거절': 6, '다른 근거': 8}
사용자 {'정답 근거': 8, '거절': 2}
사용자-거절 {'거절(맞음)': 2}
거절 {'거절(맞음)': 8, '통과(틀림)': 2}
경계 {'정답 근거': 11}
표 {'정답 근거': 10, '거절': 1}
주문 {'정답 근거': 9, '다른 근거': 2}
클래스 {'정답 근거': 9, '거절': 2, '다른 근거': 1}
장비 {'정답 근거': 7, '거절': 1}
환경 {'정답 근거': 3, '다른 근거': 1}
몬스터 {'정답 근거': 3, '거절': 1}

[직접 볼 문항]
('거절', 'How much movement does it cost to stand up after being knocked down?', 'Prone', [('Falling Off', 16), ('Climbing', 178), ('Mounting and Dismounting', 15)])
('거절', 'What happens to a creature that is surprised when combat starts?', 'Surprise', [('Surprise', 189), ('Combat', 13), ('Initiative', 13)])
('다른 근거', '매료된 상태에서 나를 홀린 상대를 공격해도 돼?', 'Charmed', 'Charm Person')
('다른 근거', '상대가 뭔가 하면 그때 반응해서 행동하려면 어떻게 해?', 'Ready', 'Reaction')
('다른 근거', '숨겨진 걸 찾으려면 어떤 행동을 해야 해?', 'Search', 'Finding Hidden Objects')
(

In [ ]:
# # 실험3. 단계별 시간 (질문 하나)
# # 설명 쓰기를 뺀 뒤라 검색 + 선택 + 검증만 잼
# # 실행하려면: 셀 전체 선택(Ctrl + A) → Ctrl + / 로 주석 해제
# question = "넘어졌을 때 일어나려면 이동력이 얼마나 들어?"
# ask3("안녕")                                       # 워밍업

# t = time.time()
# found, texts = retrieve(question)
# print("검색 (번역 2 + 검색):", round(time.time() - t, 1), "초")

# t = time.time()
# n = select(question, found)
# print("근거 선택:", round(time.time() - t, 1), "초")

# t = time.time()
# n = pick(question, found, texts, n)
# print("검증:", round(time.time() - t, 1), "초")

In [ ]:
# 실험4. 검색 갈래별 효과 (가중치 0 = 그 갈래 끔) + 예전/새 문항 나눠 보기
# 순서: [임베딩 원문, 임베딩 번역①, 임베딩 번역②, 키워드 번역①, 키워드 번역②]
# VARIANTS = {
#     "원문 임베딩만":  [1, 0, 0, 0, 0],
#     "임베딩 3개만":   [1, 1, 1, 0, 0],
#     "키워드 2개만":   [0, 0, 0, 1, 1],
#     "번역① 빼기":    [1, 0, 1, 0, 1],
#     "번역② 빼기":    [1, 1, 0, 1, 0],
#     "전체 (지금)":   SEARCH_WEIGHTS,
# }

# vranks = {}
# for name in VARIANTS:
#     vranks[name] = []

# for g in tqdm(answerable):
#     q = g["q"]
#     en1 = translate1(q)
#     en2 = translate2(q)
#     trio = semantic2([q, en1, en2], POOL_K)          # 임베딩 3갈래를 한 번에
#     lists = [trio[0], trio[1], trio[2], keyword(en1, POOL_K), keyword(en2, POOL_K)]

#     for name, weights in VARIANTS.items():
#         used = []
#         used_w = []
#         for one, w in zip(lists, weights):
#             if w > 0:                                # 꺼진 갈래는 빼고 합치기
#                 used.append(one)
#                 used_w.append(w)
#         found = merge(used, used_w, TOP_K)
#         r, _ = rank(found, g)
#         vranks[name].append(r)

# print("[전체 137문항]")
# for name in VARIANTS:
#     print(f"{name:12s}", counts(vranks[name]))

# # 예전 문항 / 새 문항 나눠 보기 (번역②가 개발셋에 맞춰졌는지 확인)
# groups = {}
# for i, g in enumerate(answerable):
#     name = "새 문항" if g.get("set") == "golden_add" else "예전 문항"
#     groups.setdefault(name, []).append(i)

# print("\n[예전 / 새 문항]")
# for name, idx in groups.items():
#     full = [vranks["전체 (지금)"][i] for i in idx]
#     no2 = [vranks["번역② 빼기"][i] for i in idx]
#     print(f"{name} ({len(idx)}문항) | 전체 {counts(full)} | 번역② 빼기 {counts(no2)}")

### 채점 개선 (Ex03)
| 후보 | 내용 | 질문당 시간 |
|---|---|---|
|정답지 형식 통일 | 같은 제목이 다른 규칙에도 있음 (Light·Shield·Resistance·Darkvision, 최대 4/137) → 모든 문항 정답을 "제목 + 쪽"으로 적고 쪽까지 비교 | 없음 |
->>>>> 데이터라벨링 필요
애매한걸 따로 세는 방법 -> 애매한 정답으로 표기

### 용어·번역 개선 (Ex05, Ex07)
| 순서 | 후보 | 내용 | 질문당 시간 |
|---|---|---|---|
| 1 | 용어표 제거 | 용어표 제거 및 번역2 절차 제거 | 없음 |
| 2 | 꼬리표 전부 | 번역② 용어를 꼬리표 5종류 41개로 확대 | 없음 |
| 3 | 번역③ | 키워드만 뽑는 번역 추가 | 거의 없음 (동시 실행) |

### 검색 개선 (Ex08, Ex09)
| 순서 | 후보 | 내용 | 질문당 시간 |
|---|---|---|---|
| 1 | 조각 중복 제거 | 같은 규칙의 조각(같은 제목·쪽)이 후보 여러 칸 차지 → 가장 높은 조각 하나만 남기기 | 없음 |
| 2 | 번역② 용어 범위 | 룰북 꼬리표 5종류 41개 전부 사용 (상태이상·행동만 고르지 않기) | 없음 |
| 3 | HyDE | 질문으로 "룰북에 있을 법한 영어 문장"을 만들어 임베딩 검색 (번역과 동시 실행) | 거의 없음 |
| 4 | 비중 조정 | [1,1,1,0.5,0.5] → 다른 조합 (BM25 0.3/0.7, 번역② 가중 상향 등) | 없음 |
| 5 | RRF 상수 | c0=60 → 20 / 100 | 없음 |
| 6 | 후보 폭 | 목록당 10개 → 20개 | 없음 |
| 7 | 번역② 프롬프트 | "확실하지 않으면 용어 넣지 말기" + 핵심 명사 유지 + 예시 1~2개 | 없음 |
| 8 | 번역③ | 키워드만 뽑는 번역 추가 | 거의 없음 (동시 실행) |
| 9 | BM25 제목 가중 | 제목 2번 → 1번 / 3번 | 없음 |
| 10 | 불용어 | 목록 확대 또는 제거 | 없음 |
| 11 | 리랭커 재시험 | 영어 번역으로 질문, 최대 1024토큰, RRF 6번째 목록으로 추가, 5등 안으로 평가 | +0.1~0.3초 |

### 근거 선택·검증 개선 (Ex12·Ex14)

**문제**: 정답이 1~2순위인데 확인(검증)에서 거절됨 (예: 성체 레드 드래곤, 몽크 기력, 추가 공격)
**원인 추정**: 확인 때 앞 800자만 보여 줌 → 긴 규칙은 답이 뒤쪽에 있어 '아니오'

| 순서 | 방안 | 내용 | 질문당 시간 | 위험 |
|---|---|---|---|---|
| 1 | 거절 직전 재확인 | 모두 실패하면 1순위만 원본 전체로 한 번 더 확인 | 거절 질문만 +0.3~0.5초 | 낮음 |
| 2 | 관련 부분 보여 주기 | 앞 800자 대신 질문과 가장 비슷한 문단을 함께 보여 줌 (키워드 점수, LLM 호출 없음) | 거의 없음 | 낮음 |
| 3 | 확인 글자 수 늘리기 | 800자 → 1500~2000자 | 확인 1번당 +0.05~0.1초 | 긴 규칙은 여전히 잘림 |
| 4 | 확인 기준 완화 | "답에 필요한 내용이 일부라도 있으면 예" | 없음 | 다른 규칙 증가 가능 |

**확인 지표 (같이 볼 것)**
- 거절 (답 있는 질문): 11 → 감소
- 다른 규칙: 11 → 유지 또는 감소
- 거절 성공 (답 없는 질문): 10/12 → 유지

### LangGraph 확장 후보 (Ex15)

**현재 그래프**: START → search → choose → check ┬ 통과 → accept → END
                                                  └ 실패 → refuse → END

| 순서 | 기법 | 하는 일 | 시간 영향 | 품질 영향 | 주의 |
|---|---|---|---|---|---|
| 1 | 그래프 그림 (`app.get_graph().draw_mermaid_png()`) | 흐름도 이미지 생성 (팀 공유·문서용) | 없음 | 없음 | - |
| 2 | 노드 캐시 (`CachePolicy`) | `search` 노드 결과 저장 → 같은 질문은 번역·검색 건너뜀 (Ex10 캐시 대체) | 반복 질문 약 0.6초 → 거의 0 | 그대로 | 청크·번역 방식을 바꾸면 반드시 캐시 비우기 (옛 결과 섞임) |
| 3 | 재시도 루프 (되돌아가는 간선 + 횟수 제한 1번) | 거절 시 다른 방식 번역(핵심 명사만)으로 한 번 더 검색 | 거절 질문만 약 +1초 | 검색 실패 6개 중 일부 회복 가능 | 모든 질문에 같은 규칙 (특정 용어 넣기 금지), 실험2로 효과·시간 같이 측정 |
| 4 | 되묻기 (`interrupt`) | 거절 시 멈추고 후보 3개 중 고르게 함 | 사람 답 대기 | 애매한 질문을 사람이 해결 | 화면(채팅 UI)이 있어야 의미 있음 |
| 5 | 진행 상황 표시 (`stream`) | 노드가 끝날 때마다 단계 표시 | 실제 시간 그대로 (체감만 줄어듦) | 그대로 | 화면이 있어야 의미 있음 |
| 6 | 분기 실행 (`Send`) | 나머지 후보 검증을 노드 여러 개로 갈라 그래프에 표시 | 그대로 (`pick`이 이미 동시 실행) | 그대로 | 동시 실행을 두 군데서 관리하게 됨 → 보류 |

### 기타 개선
| 순서 | 후보 | 내용 | 질문당 시간 |
|---|---|---|---|
| 1 | 평가용 질문 세트 분리 |
| 2 | 표 청크 다시 만들기 | 룰북 표(가격·수치)가 깨진 채로 청크에 들어가 있음 → 데이터 노트북에서 다시 생성 | 없음 |
| 3 | 실제 질문 모으기 | GM 플레이 중 들어온 규칙 질문을 저장 → 평가 세트·파인튜닝 데이터로 사용 | 없음 |
| 4 | 오류 처리 | Ollama 끊김·시간 초과 시 멈추지 않고 거절로 돌려주기 (배포 대비) | 없음 |

## 심화 핵심 기술 후보

> 기본 프로젝트 이후 RuleRAG를 한 단계 올릴 기술.

| 우선 | 기술 | 우리 코드에 적용 | 기대 효과 | 심화 주제 | 질문당 시간 |
|:---:|---|---|---|---|---|
| | **GraphRAG (규칙 연결)** | 룰북 "See also"로 규칙 그래프 생성 → 찾은 규칙과 연결된 규칙도 함께 넘기기 | 여러 규칙에 흩어진 답 해결, 다른 규칙 감소 | 답변 품질 개선 | 거의 없음 |
| | **LoRA 파인튜닝** | 규칙 고르기·확인을 작은 모델(3B~7B)로 학습 | 14B 호출 대체 | 커스텀 파인튜닝 | −0.3~0.6초 |
| | **문맥 추가 검색** | 나뉜 조각마다 "어느 규칙의 일부인지" 한 줄을 붙여 임베딩 | 긴 규칙·클래스 기능 검색 개선 | 답변 품질 개선 | 없음 (데이터 준비) |
| | 비전 LLM | 룰북 표를 이미지로 읽어 청크 다시 만들기 | 깨진 표(가격·수치) 해결 | 멀티모달 | 없음 (데이터 준비) |
| | 임베딩 파인튜닝 | LLM이 청크별 한국어 질문을 만들어 bge-m3 학습 | 번역 의존 감소 | 커스텀 파인튜닝 | 없음 |
| | 질문 분해 | 여러 규칙이 필요한 질문을 나눠서 검색 (LangGraph 분기) | 복합 질문 해결 | 고급 프롬프트·CoT | 해당 질문 +0.5~1초 |
| | LLM 심판 평가 | GM 설명이 규칙대로인지 다른 LLM이 채점 (RAGAS 방식) | 설명 품질을 숫자로 측정 | 답변 품질 평가 | 없음 (평가) |
| | 의미 캐시 | 비슷한 질문은 저장된 결과를 바로 반환 | 반복 질문 즉시 응답 | 배포 | 해당 질문 −1.3초 |
| | 구조화 출력 | Ollama `format`(JSON 스키마)으로 번호·예/아니오 받기 | 출력 읽기 오류 방지 | 고급 프롬프트 | 없음 |
| | MCP 서버 | `evidence()`를 MCP 도구로 제공 | GM·다른 에이전트가 같은 방식으로 호출 | 배포 | 없음 |

# 새 섹션